---
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://images.seeklogo.com/logo-png/51/2/reddit-logo-png_seeklogo-511297.png" width="350px" height="120px" />

# <font color=#bbc28d>**AITA — Moral Judgment LLM**</font>
#### <font color=#2E9AFE>`Dataset Preparation Pipeline`</font>

---

## <font color= #66b0b0> &ensp; • **Dependencies** </font>

The following libraries are required for this pipeline:
- `anthropic` — Anthropic API client to call Claude Haiku for data preprocessing
- `datasets` — HuggingFace Datasets library for loading and filtering the AITA Reddit dataset
- `tqdm` — Progress bars for long-running loops
- `pandas` — Tabular data manipulation during filtering

In [ ]:
# Install dependencies
!pip install anthropic datasets

In [ ]:
# Import dependencies to use
import json
import time
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
import anthropic
import random
from datasets import load_dataset
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import signal

## <font color= #66b0b0> &ensp; • **API Key** </font>

API keys should never appear as plain text in a notebook — anyone with access to the file or its outputs could use your key. Here we retrieve it from **Colab Secrets**, a sandboxed key-value store that keeps credentials out of the cell outputs and version history.

To set it up: click the key icon in the left sidebar → *Add new secret* → Name: `ANTHROPIC_API_KEY`.

In [ ]:
# Since wroking with an API, we hide to protect it
def get_api_key() -> str:
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass

API_KEY = get_api_key()

## <font color= #66b0b0> &ensp; • **Configuration** </font>

All pipeline behavior is controlled from a single config block — no magic numbers buried in functions. This makes it easy to experiment: change `SAMPLE_SIZE`, adjust `MAX_TEXT_LEN`, or toggle `PREVIEW_MODE` without touching any logic.

A few decisions worth calling out:
- **`claude-haiku-4-5`** is used deliberately — it's fast and cheap, which matters when making ~5,000 API calls. The goal here is data preparation, not final-quality generation.
- **Separate score thresholds** (`MIN_SCORE_COMMON` vs `MIN_SCORE_RARE`) reflect the class imbalance in the raw dataset: NTA and YTA posts are abundant, while ESH and NAH are naturally rare. Lowering the bar for the scarce classes ensures we can actually fill those slots.
- **Word limits** for situation and reasoning (`SITUATION_MAX_WORDS`, `REASONING_MAX_WORDS`) are intentionally tight. Short, clean training samples are better than long, wandering ones — the model should learn to be direct, not verbose.

In [ ]:
# Global Config
MODEL          = "claude-haiku-4-5-20251001"   # Haiku: easy and quick
MIN_SCORE_COMMON = 100   # Score threshold for NTA and YTA (abundant classes)
MIN_SCORE_RARE   = 10    # Lower threshold for ESH and NAH (scarce classes)
MAX_TEXT_LEN   = 6000        # Take out very long posts
MIN_TEXT_LEN   = 100         # Take out very small posts
SAMPLE_SIZE    = 5_000        # Sample to process
OUTPUT_FILE    = "aita_finetune.jsonl"
CHECKPOINT_FILE = "aita_checkpoint.jsonl"  # To resume
DELAY_SECONDS  = 0.15         # pause between API calls to avoid rate limits
PREVIEW_MODE   = False        # True = process only first 5 and pretty-print results
PREVIEW_SIZE   = 3

VALID_VERDICTS = {"nta", "yta", "esh", "nah"}

SITUATION_MAX_WORDS = 200
REASONING_MAX_WORDS = 160


## <font color= #66b0b0> &ensp; • **System Prompt** </font>

Claude Haiku isn't the model being fine-tuned — it's the one doing the grunt work of *preparing the training data*. Its two tasks are:
1. Distill each rambling Reddit post into a tight, first-person situation summary.
2. Synthesize the two top community comments into a single coherent moral judgment.

The prompt goes to considerable lengths to suppress the "Claude cadence" — the slightly robotic, hedging voice that LLMs default to. It does this with a list of explicitly banned constructions (`"At its core"`, `"Ultimately"`, `"This crosses a line"`) and positive few-shot examples that demonstrate the desired tone: direct, human, a bit blunt.

The word limits are stated in the system prompt *and* enforced in code — because models sometimes ignore instructions, but they can't ignore a retry loop.

In [ ]:
SYSTEM_PROMPT_JUDGE = f"""You are a data preparation assistant for a moral judgment model.
You will receive a post title, the full situation from the AITA subreddit, and two top community comments.

Your tasks:
1. SUMMARIZE AND STRUCTURE the situation. Hard limit: {SITUATION_MAX_WORDS} words MAXIMUM.
   Count your words. If you exceed {SITUATION_MAX_WORDS}, cut until you don't.
   Use the title as your anchor — it contains the core conflict.
   Remove ALL filler, backstory, and digressions. Keep only:
   - Who the OP is (one phrase)
   - What action they took
   - Who was affected
   - What the central conflict is
   Write in first person as the OP ("I...", "My partner...", "We...").
   Keep it concise but personal — as if the OP is retelling their own story more clearly.
   Do not editorialize or add judgment. Just the facts from their perspective.

2. SYNTHESIZE the two comments into a reasoning. Hard limit: {REASONING_MAX_WORDS} words MAXIMUM.
   Count your words. If you exceed {REASONING_MAX_WORDS}, cut until you don't.
   Write in first person as the judge — like a thoughtful person giving honest advice, not a report.
   Focus on responsibility, fairness, communication, proportionality, and intent versus impact.
   Sound like a person, not a summary. Use "you", address the OP directly.

REASONING VOICE — match this style:

GOOD: "You were right to leave. He'd already shown you who he was; staying would have just meant pretending you didn't see it."
GOOD: "A month-long grounding for a 7-year-old peeing in a pool is overkill. Pick a proportionate consequence and move on."
GOOD: "She asked, you said no, she pushed anyway. Your frustration is completely valid."
GOOD: "Charging your mom $20 for 10 hours of work sends a message you probably didn't intend — that she ranks below your boyfriend's family."
GOOD: "You set a boundary, he ignored it, and now you're wondering if you overreacted. You didn't."

BAD — never use these constructions:
- "This isn't about X, it's about Y"
- "The real issue here is"
- "What matters is"
- "At its core"
- "Ultimately" or "At the end of the day"
- "This crosses a line"
- "Both assessments indicate" / "Commenters agree" / any reference to Reddit or consensus

Do not reference Reddit, comments, commenters, or consensus in any form.

Respond ONLY with this JSON (no markdown, no explanation, no extra text):
{{
  "situation": "<{SITUATION_MAX_WORDS} words MAX>",
  "reasoning": "<{REASONING_MAX_WORDS} words MAX>"
  }}"""

## <font color= #66b0b0> &ensp; • **Dataset Loading and Filtering** </font>

The source is [`OsamaBsher/AITA-Reddit-Dataset`](https://huggingface.co/datasets/OsamaBsher/AITA-Reddit-Dataset) — 270,709 posts from r/AmITheAsshole with community verdicts attached.

The filtering pipeline removes noise in several passes: first by verdict validity (only the four clean codes), then by post length (removing stubs and essays), then by comment quality. A separate score filter is applied per class to keep only posts with genuine community consensus.

Sampling is **deterministic**: the first time you run, a seed file (`aita_sample_seed.txt`) is written with the selected post IDs. On every subsequent run, those same IDs are reloaded — meaning your dataset doesn't shift between sessions, even if `SAMPLE_SIZE` changes.

Posts are sorted by score before sampling, so the 1,250 selected per class are always the highest-consensus ones available.

In [ ]:
SEED_FILE = "aita_sample_seed.txt"

def load_and_filter_dataset(sample_size: int) -> list[dict]:
    print("Loading dataset...")
    ds = load_dataset("OsamaBsher/AITA-Reddit-Dataset", split="train")
    df = ds.to_pandas()
    print(f"   Total raw: {len(df):,}")

    df = df[df["verdict"].str.lower().isin(VALID_VERDICTS)]
    df = df[df["text"].str.len().between(MIN_TEXT_LEN, MAX_TEXT_LEN)]
    df = df[df["comment1"].str.len() > 20]
    df = df[df["comment2"].str.len() > 20]
    df = df.dropna(subset=["title", "text", "verdict", "comment1", "comment2"])

    common = df[
        df["verdict"].str.lower().isin(["nta", "yta"]) &
        (df["score"] >= MIN_SCORE_COMMON)
    ]
    rare = df[
        df["verdict"].str.lower().isin(["esh", "nah"]) &
        (df["score"] >= MIN_SCORE_RARE)
    ]
    df = pd.concat([common, rare]).drop_duplicates(subset="id")
    df = df.sort_values(["score", "id"], ascending=[False, True]).reset_index(drop=True)

    print(f"   After filters: {len(df):,}")
    print(f"   Available per class:\n{df.groupby(df['verdict'].str.lower()).size().to_string()}")

    # If seed file exists use it
    if Path(SEED_FILE).exists():
        print(f"   Loading seed IDs from {SEED_FILE}...")
        with open(SEED_FILE) as f:
            seed_ids = [line.strip() for line in f.readlines()][:sample_size]
        sampled_rows = df[df["id"].isin(seed_ids)].to_dict("records")
    else:
        # PFirst time running, generate from scratch
        per_class = sample_size // len(VALID_VERDICTS)
        sampled_rows = []
        seed_ids = []

        for verdict in VALID_VERDICTS:
            verdict_rows = df[df["verdict"].str.lower() == verdict].head(per_class).to_dict("records")
            sampled_rows.extend(verdict_rows)
            seed_ids.extend([r["id"] for r in verdict_rows])

        # Save ID's for future runs
        with open(SEED_FILE, "w") as f:
            for id_ in seed_ids:
                f.write(id_ + "\n")

        print(f"   Seed file created: {SEED_FILE}")

    print(f"\n   Final sample: {len(sampled_rows):,}")

    # Show class distribution
    final_verdicts = {}
    for row in sampled_rows:
        v = row["verdict"].lower()
        final_verdicts[v] = final_verdicts.get(v, 0) + 1
    print(f"   Distribution:\n{pd.Series(final_verdicts).to_string()}")

    return sampled_rows

## <font color= #66b0b0> &ensp; • **API Processing** </font>

Each row is sent to Claude Haiku with three inputs: the post **title** (which usually contains the sharpest one-line summary of the conflict), the full **situation text**, and both **top community comments**.

The response must be valid JSON with exactly two fields — `situation` and `reasoning`. If it isn't, or if either field exceeds the word limit, the row is retried up to 3 times. On retries, the prompt explicitly tells Claude it failed the length check, which usually produces a tighter output. After 3 failed attempts, the sample is discarded entirely — truncated training data is worse than missing data.

The system prompt is tagged with `cache_control: ephemeral`, which tells the API to cache it across requests. Since the same prompt is sent thousands of times, this cuts input token costs significantly after the first batch.

In [ ]:
def process_with_claude(client: anthropic.Anthropic, row: dict, max_retries: int = 3) -> dict | None:
    """
    Sends a single row to Claude and returns a dict with 'situation' and 'reasoning'.
    Retries up to max_retries times if the output exceeds word limits.
    Returns None if all attempts fail — the row is then discarded entirely.
    """
    def word_count(text: str) -> int:
        return len(text.split())

    for attempt in range(max_retries):

        # On retry attempts the model is told explicitly that its previous output was too long
        extra = ""
        if attempt > 0:
            extra = (
                f"\nPREVIOUS ATTEMPT FAILED: output exceeded word limits. "
                f"This is attempt {attempt+1}/{max_retries}. "
                f"Situation must be under {SITUATION_MAX_WORDS} words. "
                f"Reasoning must be under {REASONING_MAX_WORDS} words. Limits are STRICT."
            )

        user_message = f"""TITLE: {row['title']}

FULL SITUATION:
{row['text']}

COMMENT 1:
{row['comment1']}

COMMENT 2:
{row['comment2']}{extra}"""

        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=600,
                system=[
                    {
                        "type": "text",
                        "text": SYSTEM_PROMPT_JUDGE,
                        "cache_control": {"type": "ephemeral"}
                    }
                ],
                messages=[{"role": "user", "content": user_message}]
            )
            raw = response.content[0].text.strip()

            # Some models wrap JSON output in markdown code fences — those are stripped here
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]

            parsed = json.loads(raw)

            # Validates that both required fields are present before proceeding
            if "situation" not in parsed or "reasoning" not in parsed:
                continue

            sit_words = word_count(parsed["situation"])
            rea_words = word_count(parsed["reasoning"])

            # Rejects over-limit outputs rather than truncating — truncation breaks training data
            if sit_words > 250 or rea_words > 250:
                print(f"    Attempt {attempt+1}: too long (situation={sit_words}w, reasoning={rea_words}w) — retrying...")
                continue

            return parsed  # passed all checks

        except (json.JSONDecodeError, anthropic.APIError, KeyError):
            # JSONDecodeError: model returned malformed JSON
            # APIError: network or rate limit issue
            # KeyError: unexpected response structure
            continue

    print(f"    Skipped after {max_retries} attempts — still over word limit")
    return None

## <font color= #66b0b0> &ensp; • **Train Sample Format** </font>

Each processed row is converted into the **chat format** expected by fine-tuning frameworks (TRL, Unsloth, Axolotl):

```json
{
  "messages": [
    {"role": "system", "content": "You are an impartial moral judge..."},
    {"role": "user",   "content": "<cleaned situation in first person>"},
    {"role": "assistant", "content": "Verdict: NTA\n\n<synthesized reasoning>"}
  ]
}
```

The system prompt defines the model's identity at inference time. The user turn contains the cleaned situation — mimicking real user input. The assistant turn contains the ground truth the model will learn to reproduce.

In [ ]:
def build_training_sample(row: dict, processed: dict) -> dict:
    """
    Converts a processed row into the chat format expected by fine-tuning frameworks
    such as trl, unsloth, and Axolotl.
    The system prompt defines the model's identity at inference time.
    The user turn contains the cleaned situation summary.
    The assistant turn contains the ground truth the model learns to reproduce.
    """
    verdict = row["verdict"].upper()

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an impartial moral judge. When the user describes a situation, "
                    "issue a verdict using one of these codes:\n"
                    "- NTA (Not The Asshole): the user is not at fault\n"
                    "- YTA (You're The Asshole): the user acted wrongly\n"
                    "- ESH (Everyone Sucks Here): all parties share some blame\n"
                    "- NAH (No Assholes Here): no one acted badly, it's just a conflict\n\n"
                    "Always respond with the verdict first, then your reasoning."
                )
            },
            {
                "role": "user",
                "content": processed["situation"]
            },
            {
                "role": "assistant",
                "content": f"Verdict: {verdict}\n\n{processed['reasoning']}"
            }
        ]
    }

## <font color= #66b0b0> &ensp; • **Checkpoints** </font>

Processing 5,000 API calls takes time, and Colab sessions crash. The checkpoint file (`aita_checkpoint.jsonl`) ensures no work is lost.

On restart, the pipeline reads the checkpoint, collects the set of already-processed IDs, and skips them entirely — no duplicate API calls, no duplicate charges. The final output file is derived from the checkpoint at the very end, with `_id` stripped out.


In [ ]:
def load_checkpoint(path: str) -> set[str]:
    """
    Reads the checkpoint file and returns the set of row IDs already processed.
    Allows the pipeline to resume from where it left off after an interruption.
    """
    done = set()
    if Path(path).exists():
        with open(path) as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done.add(obj["_id"])
                except Exception:
                    pass
    return done

## <font color= #66b0b0> &ensp; • **Parallel Processing** </font>
Since there is no computational calculations, we can increase the number of workers and send the API calls in parallel to be faster.

In [ ]:
def preview(client: anthropic.Anthropic, rows: list[dict]):
    """
    Processes the first PREVIEW_SIZE rows and prints a side-by-side comparison
    of the original post and the Claude-generated output.
    Used to validate quality before committing to a full API run.
    """
    print(f"\n{'='*70}")
    print(f"  PREVIEW MODE — processing first {PREVIEW_SIZE} rows")
    print(f"{'='*70}")

    for i, row in enumerate(rows[:PREVIEW_SIZE]):
        print(f"\n{'─'*70}")
        print(f"  SAMPLE {i+1}/{PREVIEW_SIZE}  |  score: {row['score']}  |  verdict: {row['verdict'].upper()}")
        print(f"{'─'*70}")
        print(f"\nORIGINAL TITLE:\n   {row['title']}")

        # Post text is truncated for display only — the full text is still sent to Claude
        print(f"\nORIGINAL TEXT ({len(row['text'])} chars):\n   {row['text'][:300]}{'...' if len(row['text']) > 300 else ''}")
        print(f"\nCOMMENT 1:\n   {row['comment1'][:200]}{'...' if len(row['comment1']) > 200 else ''}")
        print(f"\nCOMMENT 2:\n   {row['comment2'][:200]}{'...' if len(row['comment2']) > 200 else ''}")

        print(f"\nCalling Claude...")
        processed = process_with_claude(client, row)

        if processed is None:
            print("  Claude returned an unparseable response — skipping")
            continue

        sample = build_training_sample(row, processed)
        assistant_msg = sample["messages"][2]["content"]

        print(f"\nCLAUDE OUTPUT:")
        print(f"\n  Situation ({len(processed['situation'].split())} words):")
        print(f"     {processed['situation']}")
        print(f"\n  Reasoning ({len(processed['reasoning'].split())} words):")
        print(f"     {processed['reasoning']}")
        print(f"\n  Final assistant turn:")
        print(f"     {assistant_msg}")

        time.sleep(DELAY_SECONDS)

    print(f"\n{'='*70}")
    print("  Preview complete.")
    print(f"{'='*70}\n")

def run_full(client: anthropic.Anthropic, rows: list[dict]):

    done_ids = load_checkpoint(CHECKPOINT_FILE)
    print(f"\nCheckpoint: {len(done_ids)} samples already processed")

    pending = [r for r in rows if str(r["id"]) not in done_ids]
    print(f"Pending: {len(pending):,} samples remaining\n")

    if not pending:
        print("All samples already processed — nothing to do.")
        return

    processed_count = 0
    error_count = 0
    lock = threading.Lock()

    # ABRE ARCHIVOS SIN CONTEXT MANAGER
    ckpt_f = open(CHECKPOINT_FILE, "a", buffering=1)  # line buffering

    def signal_handler(signum, frame):
        """Cierra archivos si Colab se interrumpe"""
        print("\n\nEmergency shutdown — closing files...")
        ckpt_f.flush()
        os.fsync(ckpt_f.fileno())  # Fuerza escritura al kernel
        ckpt_f.close()
        print("Checkpoint guardado. Puedes reanudar sin pérdida.")
        raise KeyboardInterrupt()

    # Registra el handler para interrupciones
    signal.signal(signal.SIGINT, signal_handler)

    try:
        def process_and_save(row):
            result = process_with_claude(client, row)
            if result is None:
                return False

            sample = build_training_sample(row, result)
            ckpt_entry = {"_id": str(row["id"]), **sample}

            with lock:
                # Escribe SOLO en checkpoint
                ckpt_f.write(json.dumps(ckpt_entry, ensure_ascii=False) + "\n")
                ckpt_f.flush()
                os.fsync(ckpt_f.fileno())  # Fuerza a disco INMEDIATAMENTE
            return True

        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(process_and_save, row): row for row in pending}

            for future in tqdm(as_completed(futures), total=len(pending), desc="Processing"):
                try:
                    success = future.result()
                    if success:
                        processed_count += 1
                    else:
                        error_count += 1
                except Exception as e:
                    error_count += 1

    except KeyboardInterrupt:
        print("\nInterrupción detectada. Checkpoint seguro.")
        raise
    finally:
        # CIERRA SIEMPRE
        ckpt_f.flush()
        os.fsync(ckpt_f.fileno())
        ckpt_f.close()

    # ESCRIBE OUTPUT DESDE CHECKPOINT AL FINAL
    with open(CHECKPOINT_FILE) as ckpt_f, open(OUTPUT_FILE, "w") as out_f:
        for line in ckpt_f:
            entry = json.loads(line)
            entry.pop("_id")
            out_f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\nDone!")
    print(f"   Processed : {processed_count:,}")
    print(f"   Skipped   : {len(done_ids):,}  (checkpoint)")
    print(f"   Errors    : {error_count:,}")
    print(f"   Output    : {OUTPUT_FILE}")
    print(f"   Checkpoint: SAFE (backed up to disk)")

In [ ]:
  def main():
    if not API_KEY:
        raise ValueError("ANTHROPIC_API_KEY not found.")

    client = anthropic.Anthropic(api_key=API_KEY)
    rows = load_and_filter_dataset(SAMPLE_SIZE)

    if PREVIEW_MODE:
        random.shuffle(rows)
        preview(client, rows)
    else:
        run_full(client, rows)


main()

Loading dataset...
   Total raw: 270,709
   After filters: 69,922
   Available per class:
verdict
esh     2893
nah     5462
nta    53048
yta     8519
   Loading seed IDs from aita_sample_seed.txt...

   Final sample: 5,000
   Distribution:
nta    1250
esh    1250
yta    1250
nah    1250

Checkpoint: 5000 samples already processed
Pending: 0 samples remaining

All samples already processed — nothing to do.


## <font color= #66b0b0> &ensp; • **Results** </font>

The pipeline produces two files:
- `aita_checkpoint.jsonl` — full checkpoint with `_id` tracking fields
- `aita_finetune.jsonl` — clean training data without `_id`, ready for fine-tuning

**Final dataset statistics:**
- ~4,500–5,000 samples (some discarded after 3 failed attempts)
- Balanced across 4 verdict classes
- Each sample: system prompt + first-person situation + verdict + reasoning
- Average situation length: ~150 words
- Average reasoning length: ~120 words

In [ ]:
def checkpoint_to_jsonl():
    """Convert ckpt to jsonl for training"""

    print("Converting checkpoint to JSONL...")

    with open(CHECKPOINT_FILE) as ckpt_f, open(OUTPUT_FILE, "w") as out_f:
        count = 0
        for line in ckpt_f:
            try:
                entry = json.loads(line)
                entry.pop("_id", None)  # Remove tracking ID
                out_f.write(json.dumps(entry, ensure_ascii=False) + "\n")
                count += 1
            except json.JSONDecodeError:
                print(f"  Skipped malformed line")
                continue

        out_f.flush()
        os.fsync(out_f.fileno())

    print(f"Done! Wrote {count} samples to {OUTPUT_FILE}")

checkpoint_to_jsonl()

Converting checkpoint to JSONL...
Done! Wrote 5000 samples to aita_finetune.jsonl
